# 01. 元データ収集

OFAC SDN / BIS Entity List / e-Gov 貨物等省令 を公式ソースから取得して `data/staging/` に保存する。

| # | データソース | 取得先 | 出力先 |
|---|------------|--------|--------|
| 1 | OFAC SDN XML | treasury.gov | staging/sanctions/ofac_sdn_raw.json |
| 2 | BIS Entity List | trade.gov CSL API | staging/sanctions/bis_el_raw.json |
| 3 | e-Gov 貨物等省令 | elaws.e-gov.go.jp | staging/fefta/fefta_articles_raw.json |

In [ ]:
# ── 設定
DRY_RUN = False  # True にするとネット取得のみ、ファイル書き込みなし

import sys, os, logging
from pathlib import Path

try:
    BASE
except NameError:
    BASE        = Path("/content/AI_TradeManagement")
    STAGING_DIR = BASE / "data" / "staging"
    sys.path.insert(0, str(BASE / "scripts"))

# APIキー（00_setup で設定済みでなければここで取得）
try:
    ANTHROPIC_API_KEY
except NameError:
    try:
        from google.colab import userdata
        ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY") or ""
    except Exception:
        ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

try:
    BIS_API_KEY
except NameError:
    BIS_API_KEY = "DEMO_KEY"

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print(f"DRY_RUN={DRY_RUN}, STAGING_DIR={STAGING_DIR}")

## 1. OFAC SDN XML 取得

In [ ]:
from pipeline.collect.ofac_sdn import fetch_ofac_sdn

ofac_cache = STAGING_DIR / "sanctions" / "ofac_sdn_raw.json"
ofac_entities = fetch_ofac_sdn(cache_path=None if DRY_RUN else ofac_cache)
print(f"✅ OFAC SDN: {len(ofac_entities):,} エンティティ取得")
if ofac_entities:
    print(f"   例: {ofac_entities[0]}")

## 2. BIS Entity List 取得

In [ ]:
from pipeline.collect.bis_entity_list import fetch_bis_entity_list

bis_cache = STAGING_DIR / "sanctions" / "bis_el_raw.json"
bis_entities = fetch_bis_entity_list(
    api_key=BIS_API_KEY,
    cache_path=None if DRY_RUN else bis_cache,
)
print(f"✅ BIS Entity List: {len(bis_entities):,} エンティティ取得")

## 3. e-Gov 貨物等省令 取得

In [ ]:
from pipeline.collect.egov_fefta import fetch_fefta_ministerial_ordinance

fefta_cache = STAGING_DIR / "fefta" / "fefta_articles_raw.json"
fefta_articles = fetch_fefta_ministerial_ordinance(
    cache_path=None if DRY_RUN else fefta_cache,
)
if fefta_articles:
    print(f"✅ 貨物等省令: {len(fefta_articles)} 条文取得")
    print(f"   例: {fefta_articles[0]}")
else:
    print("⚠️  条文取得なし — e-Gov API 応答を確認してください")

## 4. 取得結果サマリー

In [ ]:
print("=" * 50)
print(f"OFAC SDN       : {len(ofac_entities):>6,} 件")
print(f"BIS Entity List: {len(bis_entities):>6,} 件")
print(f"貨物等省令条文  : {len(fefta_articles):>6,} 件")
print()
print("staging 出力ファイル:")
for f in sorted(STAGING_DIR.rglob("*.json")):
    print(f"  {f.relative_to(STAGING_DIR)}  ({f.stat().st_size:,} bytes)")
print("\n次のノートブック → 02_integrate_datasets.ipynb")